# AF2·05 — Residue Frames & the Structure Module

**Mechanism of the day:** stop describing a protein by pairwise distances and start
*placing it in space*. AlphaFold's structure module represents a protein as a **gas of
rigid frames** — one local coordinate system per residue — and everything from here on
operates on those frames.

You met this object twice already: the SE(3) frame in `0.1` (built from three backbone
atoms by Gram-Schmidt) and frame *generation* in `1.6`. Now it becomes the backbone of
AlphaFold's output head.

Why frames instead of raw `(x, y, z)` coordinates?

- **A frame carries orientation, not just position.** Rotation `R_i` plus translation
  `t_i` tells you where residue `i` sits *and* which way it faces — enough to place every
  backbone atom (and later every sidechain atom) from fixed ideal local coordinates.
- **Frames make invariance free.** The physically meaningful quantities are *relative*
  frames — residue `j` as seen from residue `i`'s local viewpoint. Those are unchanged
  when you rotate or translate the whole protein. Reasoning in relative frames means the
  network never has to commit to a global coordinate system, which is exactly the
  property (rung 06's Invariant Point Attention) that makes the structure module respect
  the symmetry of 3D space.

The structure module starts from a **"black hole" initialization** — every residue's
frame at the identity rotation and the origin — and iteratively moves the frames until
they spell out a structure. This notebook builds the frame machinery those iterations
rely on; rung 06 builds the attention that drives them; rung 07 builds the loss.

**How to use this notebook:** implement the reps, make the checkpoints pass. Pure
geometry — no training, runs instantly. Solutions at the bottom.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

torch.manual_seed(0); rng = np.random.default_rng(0)
plt.rcParams['axes.spines.top'] = False; plt.rcParams['axes.spines.right'] = False
BLUE, GREEN, INK, RED = '#2a78d6', '#008300', '#52514e', '#e34948'
L = 16

# ideal backbone atoms in a residue's LOCAL frame (CA at the origin). These particular
# values are chosen so that a residue built from them has the identity frame — i.e. the
# local frame and the global frame coincide when a residue sits at the origin unrotated.
N_LOCAL  = torch.tensor([-0.5, 1.4, 0.0])
CA_LOCAL = torch.tensor([0.0, 0.0, 0.0])
C_LOCAL  = torch.tensor([1.52, 0.0, 0.0])

def rand_rotations(n):
    '''n uniformly random rotation matrices (via QR).'''
    A = torch.randn(n, 3, 3); Q, S = torch.linalg.qr(A)
    Q = Q * torch.sign(torch.diagonal(S, dim1=-2, dim2=-1))[:, None, :]
    Q[torch.det(Q) < 0, :, 0] *= -1
    return Q
print('a residue frame = (R [3,3] rotation, t [3] translation); protein length L =', L)

## Part 1 — a residue frame from three atoms

The frame for residue `i` is built from its backbone `N`, `Cα`, `C` atoms by
Gram-Schmidt — identical to `0.1`, restated here because it is the atom-to-frame map the
whole module rests on:

- `t = Cα` (the origin of the local frame),
- `e1 = (C − Cα)` normalized,
- `e2 =` the part of `(N − Cα)` orthogonal to `e1`, normalized,
- `e3 = e1 × e2`,
- `R = [e1 | e2 | e3]` as **columns**.

`R` is a proper rotation and `(R, t)` maps this residue's local coordinates into global
space.

### Rep 1 — `rigid_from_three_points(N, CA, C)`
Return `(R, t)`: the `[3,3]` rotation (axes as columns) and `[3]` translation. All
inputs are `[3]` vectors.

In [ ]:
def rigid_from_three_points(N, CA, C):
    '''Gram-Schmidt residue frame: returns (R [3,3] with axes as columns, t [3]).'''
    # YOUR CODE HERE
    # hint: e1 = normalize(C - CA); v = N - CA; e2 = normalize(v - (e1@v)*e1)
    #       e3 = cross(e1, e2); R = stack([e1, e2, e3], dim=-1); t = CA
    raise NotImplementedError

# --- checkpoint ---
R, t = rigid_from_three_points(N_LOCAL, CA_LOCAL, C_LOCAL)
assert torch.allclose(R @ R.T, torch.eye(3), atol=1e-5), 'R must be orthonormal'
assert torch.isclose(torch.det(R), torch.tensor(1.0), atol=1e-5), 'R must be a proper rotation'
# by construction the ideal local atoms yield the identity frame at the origin
assert torch.allclose(R, torch.eye(3), atol=1e-5) and torch.allclose(t, torch.zeros(3), atol=1e-5)
print('residue frame ok — ideal local atoms give the identity frame (as designed) ✓')

## Part 2 — seeing the world from a residue's viewpoint

The frame lets you translate between **global** coordinates and a residue's **local**
coordinates. Both directions are one line:

$$
\begin{aligned} \text{global}\to\text{local:}&\quad \text{local} = R^\top (p - t) \\ \text{local}\to\text{global:}&\quad p = R\cdot\text{local} + t \end{aligned}
$$

Invariant Point Attention (rung 06) lives on this: each residue proposes points in its
*own* local frame, and to compare them it maps everything through these transforms. That
is why the comparison ends up independent of the global pose.

### Rep 2 — `to_local(R, t, p)`
Express global point(s) `p` `[..., 3]` in the frame `(R, t)`. Return `Rᵀ (p − t)`.
(With `R`'s axes as columns, `Rᵀ x` is `(x @ R)`.)

In [ ]:
def to_local(R, t, p):
    '''Global points p [...,3] -> the frame's local coordinates.'''
    # YOUR CODE HERE
    # hint: (p - t) @ R      # equals R^T (p - t) since R's columns are the axes
    raise NotImplementedError

# --- checkpoint ---
R = rand_rotations(1)[0]; t = torch.randn(3)
p = torch.randn(7, 3)
loc = to_local(R, t, p)
assert loc.shape == (7, 3)
# the frame's own origin maps to the local origin
assert torch.allclose(to_local(R, t, t[None]), torch.zeros(1, 3), atol=1e-5)
# distances are preserved (it is a rigid transform)
assert torch.allclose((loc[0] - loc[1]).norm(), (p[0] - p[1]).norm(), atol=1e-5)
print('to_local ok — rigid change of viewpoint, distances preserved ✓')

### Rep 3 — `to_global(R, t, local)`
The inverse: place local point(s) back into global space, `R · local + t`.

In [ ]:
def to_global(R, t, local):
    '''Local coordinates -> global points.'''
    # YOUR CODE HERE
    # hint: local @ R.T + t
    raise NotImplementedError

# --- checkpoint ---
back = to_global(R, t, to_local(R, t, p))
assert torch.allclose(back, p, atol=1e-5), 'to_global must invert to_local'
# placing the ideal local atoms with a frame reproduces that residue's atoms
Rf, tf = rand_rotations(1)[0], torch.randn(3)
Cg = to_global(Rf, tf, C_LOCAL[None])[0]
assert torch.allclose(Cg, Rf @ C_LOCAL + tf, atol=1e-5)
print('to_global ok — round-trips to_local exactly ✓')

## Part 3 — relative frames, and why global pose stops mattering

Here is the payoff of the whole representation. The relationship between two residues is
their **relative frame** — residue `j` expressed in residue `i`'s local frame:

$$
R_{\text{rel}} = R_i^\top R_j, \qquad t_{\text{rel}} = R_i^\top (t_j - t_i)
$$

Now rotate and translate the *entire* protein by some global `(R_g, t_g)`. Every frame
changes. But the **relative** frame between any two residues does **not** — the `R_g`s
cancel. So any function of relative frames is automatically invariant to global pose.
That is the mathematical reason AlphaFold's structure module never has to choose an
orientation for the protein, and why its predictions transform correctly with the input.

### Rep 4 — `relative_frame(Ri, ti, Rj, tj)`
Return `(R_rel, t_rel) = (R_iᵀ R_j,  R_iᵀ (t_j − t_i))`.

In [ ]:
def relative_frame(Ri, ti, Rj, tj):
    '''Frame j expressed in frame i: (R_i^T R_j, R_i^T (t_j - t_i)).'''
    # YOUR CODE HERE
    # hint: R_rel = Ri.transpose(-1,-2) @ Rj ; t_rel = (tj - ti) @ Ri
    raise NotImplementedError

# --- checkpoint ---
Rs = rand_rotations(L); ts = torch.cumsum(torch.randn(L, 3) * 1.5, 0)   # a toy set of frames
i, j = 3, 11
Rrel, trel = relative_frame(Rs[i], ts[i], Rs[j], ts[j])
assert Rrel.shape == (3, 3) and trel.shape == (3,)
# apply a global rotation+translation to every frame, recompute -> relative frame unchanged
Rg = rand_rotations(1)[0]; tg = torch.randn(3)
Rs_g = torch.einsum('ij,njk->nik', Rg, Rs); ts_g = ts @ Rg.T + tg
Rrel2, trel2 = relative_frame(Rs_g[i], ts_g[i], Rs_g[j], ts_g[j])
assert torch.allclose(Rrel, Rrel2, atol=1e-5), 'relative rotation must be pose-invariant'
assert torch.allclose(trel, trel2, atol=1e-5), 'relative translation must be pose-invariant'
print('relative frame ok — invariant to global rotation+translation ✓')
print('|t_rel| (distance between the two residues, in frame i) = %.2f, unchanged after moving' % trel.norm())

## Part 4 — the residue gas: frames ⇄ a backbone

Put it together. A set of `L` frames *is* a backbone: place the ideal local `N, Cα, C`
atoms through each residue's frame and you have global coordinates. Going the other way,
Gram-Schmidt recovers the frames from the atoms. The structure module's job (rungs 06–07)
is to *move the frames*; this reconstruction is how those frames become atoms you can
score and see.

### Rep 5 — `build_backbone(Rs, ts)`
Given per-residue frames `Rs` `[L,3,3]` and `ts` `[L,3]`, place the ideal
`N_LOCAL, CA_LOCAL, C_LOCAL` atoms through each frame. Return `[L, 3, 3]` — for each
residue, the global `(N, Cα, C)` coordinates.

In [ ]:
def build_backbone(Rs, ts):
    '''Frames -> global backbone atoms [L, 3atoms, 3coords] (order N, CA, C).'''
    # YOUR CODE HERE
    # hint: ideal = torch.stack([N_LOCAL, CA_LOCAL, C_LOCAL])   # [3,3]
    #       for each residue: to_global(Rs[i], ts[i], ideal)   (or vectorize with einsum)
    raise NotImplementedError

# --- checkpoint ---
bb = build_backbone(Rs, ts)
assert bb.shape == (L, 3, 3), 'expected [L, 3 atoms, 3 coords]'
# CA atom of each residue must equal that residue's translation
assert torch.allclose(bb[:, 1, :], ts, atol=1e-5), 'the CA atom is the frame origin'
# round-trip: recover frames from the built atoms
R_rec, t_rec = zip(*[rigid_from_three_points(bb[i, 0], bb[i, 1], bb[i, 2]) for i in range(L)])
assert torch.allclose(torch.stack(R_rec), Rs, atol=1e-4), 'frames must round-trip through atoms'
print('build_backbone ok — frames <-> atoms round-trips exactly ✓')

In [ ]:
# Two views of the same protein: original, and globally rotated+translated. The frames
# and coordinates differ entirely; the internal geometry (relative frames) is identical.
bb = build_backbone(Rs, ts)
bb_moved = build_backbone(torch.einsum('ij,njk->nik', Rg, Rs), ts @ Rg.T + tg)

def draw(ax, atoms, Rframes, title, col):
    CA = atoms[:, 1, :].numpy()
    ax.plot(CA[:, 0], CA[:, 1], CA[:, 2], '-', color=col, lw=1.5)
    ax.scatter(CA[:, 0], CA[:, 1], CA[:, 2], s=14, color=col)
    for i in range(0, L, 2):        # draw each residue's local axes as little quivers
        for a, c in zip(range(3), [RED, GREEN, BLUE]):
            v = Rframes[i][:, a].numpy() * 1.2
            ax.quiver(CA[i, 0], CA[i, 1], CA[i, 2], v[0], v[1], v[2], color=c, lw=1)
    ax.set_title(title, fontsize=10); ax.set_xticklabels([]); ax.set_yticklabels([]); ax.set_zticklabels([])

fig = plt.figure(figsize=(10, 4.2))
ax1 = fig.add_subplot(121, projection='3d'); draw(ax1, bb, Rs, 'a toy backbone (with residue frames)', INK)
ax2 = fig.add_subplot(122, projection='3d')
draw(ax2, bb_moved, torch.einsum('ij,njk->nik', Rg, Rs), 'same protein, globally moved', BLUE)
plt.tight_layout(); plt.show()

i, j = 3, 11
r1 = relative_frame(Rs[i], ts[i], Rs[j], ts[j])
Rs_g = torch.einsum('ij,njk->nik', Rg, Rs); ts_g = ts @ Rg.T + tg
r2 = relative_frame(Rs_g[i], ts_g[i], Rs_g[j], ts_g[j])
print('relative frame (residue %d as seen from %d): identical across the two poses?  %s'
      % (j, i, torch.allclose(r1[0], r2[0], atol=1e-5) and torch.allclose(r1[1], r2[1], atol=1e-5)))
print('The pictures look different; the internal geometry the network reasons about does not. ✓')

## Reflection — what just transferred

- **A protein is a gas of rigid frames**, one per residue: a rotation and a translation
  that carry both *where* a residue is and *how it is oriented* — enough to place all its
  atoms from fixed ideal local coordinates.
- **`to_local` / `to_global`** move points between a residue's viewpoint and the shared
  world. Every geometric operation in the structure module is phrased with them.
- **Relative frames are pose-invariant.** `R_iᵀ R_j` and `R_iᵀ(t_j − t_i)` do not change
  when you rotate or translate the whole protein — the single most important fact in the
  structure module, and the reason it needs no canonical orientation.
- **Frames ⇄ atoms** is a clean round-trip: build the backbone by placing ideal atoms
  through the frames; recover the frames by Gram-Schmidt. The module *moves frames*; this
  turns them into coordinates.
- The structure module starts from the **black-hole init** (all frames at identity/origin)
  and edits the frames toward the answer — which is exactly what the next two rungs build.

**Next rung:** `AF2·06 — Invariant Point Attention`. Residues attend to each other, but
the queries, keys, and values include *points in each residue's local frame*. Because the
comparison happens through the relative-frame transforms you just built, the whole
attention is provably invariant to global rotation and translation — geometry-aware
attention, done right.

---
Scroll down only after you've done the reps.

## Solutions appendix (peek only after trying)

In [ ]:
def rigid_from_three_points(N, CA, C):
    e1 = C - CA; e1 = e1 / e1.norm()
    v = N - CA; e2 = v - (e1 @ v) * e1; e2 = e2 / e2.norm()
    e3 = torch.cross(e1, e2, dim=-1)
    R = torch.stack([e1, e2, e3], dim=-1)      # axes as columns
    return R, CA

def to_local(R, t, p):
    return (p - t) @ R                          # R^T (p - t)

def to_global(R, t, local):
    return local @ R.T + t

def relative_frame(Ri, ti, Rj, tj):
    return Ri.transpose(-1, -2) @ Rj, (tj - ti) @ Ri

def build_backbone(Rs, ts):
    ideal = torch.stack([N_LOCAL, CA_LOCAL, C_LOCAL])          # [3 atoms, 3]
    return torch.einsum('nij,aj->nai', Rs, ideal) + ts[:, None, :]

print('reference solutions loaded — re-run the checkpoint cells above')